# SOXL paper/sim replay production candidate v1

目标：把已经验证的 `schedule -> dashboard -> paper intent` 链路固定成 production 候选 notebook。

边界：v1 只运行 paper/sim，不连接真实 broker，不触发真实下单。生产晋升前仍需更严格模型、风控和外部调度。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api import (
    PaperBrokerAdapter,
    build_target_position_intent_from_dashboard,
    execute_paper_intent,
    write_intent_ledger,
)
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('production/paper_sim_replay_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
config = ModelDashboardPipelineConfig(
    name='production_paper_sim_replay_v1',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
    dashboard_top_n=5,
    notify_details=True,
    retries=1,
)
config

ModelDashboardPipelineConfig(name='production_paper_sim_replay_v1', card_dir=PosixPath('production/paper_sim_replay_v1/matrix_card'), dashboard_output_path=PosixPath('production/paper_sim_replay_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'random_forest_clf'), top_n_factors=3, dashboard_top_n=5, attribution_card_dir=None, dashboard_page_output_path=None, dashboard_page_title='SOXL Dashboard', analysis_manifest_preflight=False, analysis_manifest_auto_backfill=False, analysis_manifest_dataset_name='schedule_dashboard_refresh', analysis_manifest_universe=('SOXL',), analysis_manifest_calendar_path='', notify_details=True, retries=1)

In [2]:
schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
)
schedule_result

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


JobResult(name='production_paper_sim_replay_v1', status='success', payload={'matrix_card': 'production/paper_sim_replay_v1/matrix_card', 'dashboard_export': 'production/paper_sim_replay_v1/dashboard_export.json', 'dashboard_schema_version': 'dashboard.model_matrix.v1', 'dashboard_artifact_id': 'matrix_card', 'dashboard_manifest_kind': 'analysis_manifest', 'dashboard_manifest_path': 'production/paper_sim_replay_v1/matrix_card/analysis_manifest.json', 'dashboard_manifest_hash': '7fb66f5ca4f3262ed4d4947615390dcd5a3daa9b874e00fe69f55792c3014174', 'dashboard_manifest_producer': 'src.analysis.exports', 'dashboard_page': '', 'dashboard_page_schema_version': '', 'dashboard_view_model_schema_version': '', 'dashboard_page_model_card_count': 0, 'top_model': 'ridge_mse__ic_weighted_composite', 'attribution_card': '', 'attribution_status': 'not_provided', 'matrix_status': 'feature_set_matrix_completed', 'variant_count': 6, 'top_failure_mode': 'return_stability_lag', 'top_failure_segment': 'stress',

In [3]:
dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
intent = build_target_position_intent_from_dashboard(
    dashboard_payload,
    instrument='SOXL',
    min_position=0.0,
    max_position=0.75,
)
intent.to_dict()

{'instrument': 'SOXL',
 'target_position': 0.75,
 'source_model': 'ridge_mse__ic_weighted_composite',
 'source_artifact': 'matrix_card',
 'mode': 'paper',
 'schema_version': 'api.execution_intent.v1',
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
 'metadata': {'execution_gate': 'paper_review',
  'raw_avg_position': 0.7518156857658157,
  'min_position': 0.0,
  'max_position': 0.75,
  'base_model': 'ridge_mse',
  'feature_set': 'ic_weighted_composite',
  'stability_score': 29.940062203007624,
  'max_drawdown_pct': 29.020142371836933,
  'promotion_decision': 'promote_candidate',
  'promotion_score': 29.940062203007624,
  'promotion_blockers': '',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'analysis_overview_schema_version': 'analysis.overview.v1',
  'analysis_top_weakness_model': 'ridge_mse__all_features',
  'analysis_top_weakness_segment': 'stress',
  'analysis_top_weakness_decision_hint': 'block_or_rework',
  'analysis_top_gap_segment': 'downtrend',
  'analys

In [4]:
broker = PaperBrokerAdapter(cash=10000.0, shares=0)
execution_result = execute_paper_intent(intent, broker=broker, price=100.0)
(CARD / 'execution_intent.json').write_text(json.dumps(intent.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'paper_execution_result.json').write_text(json.dumps(execution_result.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
write_intent_ledger(CARD / 'intent_ledger.json', intent=intent, result=execution_result)
execution_result.to_dict()

{'schema_version': 'api.paper_execution_result.v1',
 'status': 'paper_executed',
 'broker_name': 'paper',
 'intent': {'instrument': 'SOXL',
  'target_position': 0.75,
  'source_model': 'ridge_mse__ic_weighted_composite',
  'source_artifact': 'matrix_card',
  'mode': 'paper',
  'schema_version': 'api.execution_intent.v1',
  'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
  'metadata': {'execution_gate': 'paper_review',
   'raw_avg_position': 0.7518156857658157,
   'min_position': 0.0,
   'max_position': 0.75,
   'base_model': 'ridge_mse',
   'feature_set': 'ic_weighted_composite',
   'stability_score': 29.940062203007624,
   'max_drawdown_pct': 29.020142371836933,
   'promotion_decision': 'promote_candidate',
   'promotion_score': 29.940062203007624,
   'promotion_blockers': '',
   'dashboard_schema_version': 'dashboard.model_matrix.v1',
   'analysis_overview_schema_version': 'analysis.overview.v1',
   'analysis_top_weakness_model': 'ridge_mse__all_features',
   'analysis_top_wea

In [5]:
intent_metadata = dict(intent.metadata or {})
analysis_overview = dashboard_payload.get('analysis_overview', {})
top_gap = (analysis_overview.get('largest_relative_return_spread_segments') or [{}])[0]
summary = {
    'schema_version': 'production.paper_sim_replay.v1',
    'status': 'paper_replay_completed',
    'mode': intent.mode,
    'live_broker_enabled': False,
    'instrument': intent.instrument,
    'source_model': intent.source_model,
    'target_position': intent.target_position,
    'execution_gate': intent_metadata.get('execution_gate', ''),
    'promotion_decision': intent_metadata.get('promotion_decision', ''),
    'promotion_blockers': intent_metadata.get('promotion_blockers', ''),
    'analysis_overview_schema': intent_metadata.get('analysis_overview_schema_version', ''),
    'analysis_top_gap_segment': intent_metadata.get('analysis_top_gap_segment', ''),
    'analysis_top_gap_spread': intent_metadata.get('analysis_top_gap_spread', 0.0),
    'analysis_top_gap_best_model': intent_metadata.get('analysis_top_gap_best_model', ''),
    'analysis_top_gap_worst_model': intent_metadata.get('analysis_top_gap_worst_model', ''),
    'analysis_top_weakness_model': intent_metadata.get('analysis_top_weakness_model', ''),
    'analysis_top_weakness_decision_hint': intent_metadata.get('analysis_top_weakness_decision_hint', ''),
    'backtest_schema_version': intent_metadata.get('backtest_schema_version', ''),
    'backtest_walk_forward_fold_count': intent_metadata.get('backtest_walk_forward_fold_count', 0),
    'backtest_slippage_bps': intent_metadata.get('backtest_slippage_bps', 0.0),
    'paper_shares': execution_result.snapshot.shares,
    'paper_position': execution_result.snapshot.position,
    'schedule_status': schedule_result.status,
    'schedule_attempts': schedule_result.payload.get('attempts'),
    'notifications': notifications,
    'dashboard_export': str(CARD / 'dashboard_export.json'),
    'intent_ledger': str(CARD / 'intent_ledger.json'),
    'matrix_card': str(CARD / 'matrix_card'),
    'guardrail': intent.guardrail,
    'analysis_overview': analysis_overview,
    'top_gap': top_gap,
}
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'production.paper_sim_replay.v1',
 'status': 'paper_replay_completed',
 'mode': 'paper',
 'live_broker_enabled': False,
 'instrument': 'SOXL',
 'source_model': 'ridge_mse__ic_weighted_composite',
 'target_position': 0.75,
 'execution_gate': 'paper_review',
 'promotion_decision': 'promote_candidate',
 'promotion_blockers': '',
 'analysis_overview_schema': 'analysis.overview.v1',
 'analysis_top_gap_segment': 'downtrend',
 'analysis_top_gap_spread': 47.970097543991834,
 'analysis_top_gap_best_model': 'random_forest_clf__ic_weighted_composite',
 'analysis_top_gap_worst_model': 'ridge_mse__all_features',
 'analysis_top_weakness_model': 'ridge_mse__all_features',
 'analysis_top_weakness_decision_hint': 'block_or_rework',
 'backtest_schema_version': 'backtest.config.v1',
 'backtest_walk_forward_fold_count': 5,
 'backtest_slippage_bps': 1.0,
 'paper_shares': 75,
 'paper_position': 0.75,
 'schedule_status': 'success',
 'schedule_attempts': 1,
 'notifications': ['production_pa

In [6]:
assert summary['execution_gate'] in {'paper_review', 'blocked_to_min_position'}
assert summary['analysis_overview_schema'] == 'analysis.overview.v1'
assert summary['promotion_decision']
assert summary['backtest_schema_version'] == 'backtest.config.v1'
assert summary['backtest_walk_forward_fold_count'] >= 1
assert 'regime_gap=' in ' '.join(summary['notifications'])
assert intent_metadata['execution_gate'] == summary['execution_gate']

{
    'dashboard_top_models': dashboard_payload['top_models'][:3],
    'feature_set_summary': dashboard_payload['feature_set_summary'],
    'execution_gate': summary['execution_gate'],
    'promotion_decision': summary['promotion_decision'],
    'analysis_top_gap_segment': summary['analysis_top_gap_segment'],
    'backtest_schema_version': summary['backtest_schema_version'],
    'ledger_exists': (CARD / 'intent_ledger.json').exists(),
    'live_broker_enabled': summary['live_broker_enabled'],
}

{'dashboard_top_models': [{'rank': 1,
   'model': 'ridge_mse__ic_weighted_composite',
   'base_model': 'ridge_mse',
   'feature_set': 'ic_weighted_composite',
   'return_pct': 38.31176843102375,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -19.806612732919724,
   'max_drawdown_pct': 29.020142371836933,
   'sharpe': 3.115575854413103,
   'avg_position': 0.7518156857658157,
   'turnover': 1.6261434036754974,
   'stability_score': 29.940062203007624},
  {'rank': 2,
   'model': 'ridge_mse__top_ic_factors',
   'base_model': 'ridge_mse',
   'feature_set': 'top_ic_factors',
   'return_pct': 37.09597053157543,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -21.022410632368043,
   'max_drawdown_pct': 29.12731437649879,
   'sharpe': 3.0519794135557325,
   'avg_position': 0.7473726988299527,
   'turnover': 1.949158985539,
   'stability_score': 27.308303725431273},
  {'rank': 3,
   'model': 'random_forest_clf__ic_weighted_composite',
   'base_m

## 中文结论

- 做了什么：运行 production 候选链路，完成 schedule 刷新、dashboard export、paper intent 生成和 PaperBrokerAdapter 模拟执行。
- 核心指标：source model、target_position、paper shares、paper position、execution_gate、promotion_decision 和 backtest schema 均写入 `production_summary.json` 与 `execution_intent.json`。
- regime 差异：dashboard export 保留 `analysis_overview.v1`，paper intent metadata 记录最大 `regime_gap`、top weakness 和 promotion 决策，生产回放不再只看全区间仓位。
- 是否晋升：否。本 notebook 是 paper/sim production candidate，不是 live 生产策略。
- 风险：`live_broker_enabled=False`；guardrail=`v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。`。后续需要严格模型实现、风控阈值、外部调度和审计后才能讨论 live adapter。
- 下一步：补 manifest 化生产注册表、外部调度入口和更严格的模型 adapter 替换计划。